In [ ]:
import torch
from tqdm import tqdm
import model
import matplotlib.pyplot as plt
from nerf_dataset_simple import load_blender_data, get_rays

import torch.nn.functional as F
Model = model.MLP(33, 4, [64, 64, 128, 64])

# torch.save(Model.state_dict(), 'nerf_lego_final.pth') 可以把训练好的模型放进 nerf_lego_final.pth

checkpoint = torch.load('nerf_test.pth', map_location='cpu') # 调用已经训练好的模型
Model.load_state_dict(checkpoint)

Model.eval()
print("模型权重加载成功，准备开始渲染！")

# 1. 加载测试数据
print("Loading test data...")
images_test, poses_test, hwf_test = load_blender_data('data/lego', split='test')
H, W, focal = hwf_test
print(f"Loaded {len(images_test)} test images.")

# 2. 选择一个要渲染的视角
view_index_to_render = 30
target_pose = torch.from_numpy(poses_test[view_index_to_render])
print(f"Rendering view {view_index_to_render}...")

# 3. 生成该视角下的所有光线
rays_o, rays_d = get_rays(H, W, focal, target_pose)
rays_o = rays_o.reshape(-1, 3)
rays_d = rays_d.reshape(-1, 3)

# 4. 分批次渲染，避免内存溢出
chunk_size = 1024
rgb_chunks = []

with torch.no_grad():
    for i in tqdm(range(0, rays_o.shape[0], chunk_size)):
        # 获取当前批次的光线
        batch_rays_o = rays_o[i:i+chunk_size]
        batch_rays_d = rays_d[i:i+chunk_size]
        rgb_map= model.Calc_Light(batch_rays_o, batch_rays_d, Model)
        rgb_map = torch.pow(torch.clamp(rgb_map, 0.0, 1.0), 1.0 / 2.2) # 调整颜色
        rgb_chunks.append(rgb_map.cpu())

# 5. 拼接并显示图像
full_image = torch.cat(rgb_chunks, dim=0).reshape(H, W, 3)

plt.figure(figsize=(8, 8))
plt.imshow(full_image)
plt.title(f"Rendered Image (View {view_index_to_render})")
plt.axis('off')
plt.show()

# 同时显示真实图像作为对比
plt.figure(figsize=(8, 8))
ground_truth_image = images_test[view_index_to_render]
plt.imshow(ground_truth_image)
plt.title(f"Ground Truth (View {view_index_to_render})")
plt.axis('off')
plt.show()
